# TinyML - Causal Trees (CT)

**Author:** Thommas Kevin Sales Flores  
**Institution:** Federal University of Rio Grande do Norte  
**Email:** thommas.flores@ufrn.br

---
This notebook demonstrates how to build, train, and export **Causal Trees** to embedded hardware (Arduino / ESP32).  
All examples use an honest single Causal Tree that estimates Heterogeneous Treatment Effects (HTE).

### Supported building blocks

| Component | Options |
|-----------|---------|
| **Task** | `regression`, `binary`, `multiclass` |
| **Split criterion** | `variance`, `mse`, `tau_risk` |
| **Estimation** | `honest=True` (default) or adaptive |
| **Metrics** | `tau_risk`, `ate_bias`, `mse`, `mae`, `rmse`, `bce`, `accuracy`, `risk_diff_error`, `scce`, `accuracy_multi`, `macro_ate_error` |
| **VI / Policy** | `elbo_reg`, `ltc_reg_loss`, `elbo_binary`, `policy_loss_bin`, `elbo_multi`, `policy_loss_multi`, `dr_ate` |


## Environment Setup

Uncomment and run the cell below the first time to install dependencies.


In [1]:
#!pip install numpy matplotlib scikit-learn scipy torch

## 1. Setup and Imports


In [2]:
import sys, os
sys.path.append('36_CT')   # adjust if running from a different directory

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs
from sklearn.model_selection import train_test_split

from model  import CausalTreeModel
from layers import get_criterion
from losses import compute_metric, METRIC_NAMES
from utils  import (
    export_to_json,
    train_causal_tree,
    plot_tree,
    print_tree,
    plot_cate_distribution,
    plot_treatment_effect_by_feature,
    plot_decision_boundary,
    plot_leaf_effects,
    plot_multiclass_effects,
    plot_training_history,
)
from vi import compute_vi, doubly_robust_ate
from cpp_generator import generate_ino

os.makedirs('json_model',   exist_ok=True)
os.makedirs('arduino_code', exist_ok=True)
os.makedirs('figures',      exist_ok=True)

print('Available metrics:', METRIC_NAMES)


Available metrics: ['mse', 'mae', 'rmse', 'tau_risk', 'ate_bias', 'bce', 'accuracy', 'risk_diff_error', 'scce', 'accuracy_multi', 'macro_ate_error']


---
## 2. Example 1 — Regression: Continuous Outcome

We train a Causal Tree to estimate the CATE for a continuous outcome where the true
effect is a step function of $X_0$:

$$\tau^*(x) = \begin{cases} 2.0 & X_0 > 0 \\ 0.5 & X_0 \le 0 \end{cases}$$

The honest tree with the **Variance criterion** automatically discovers the boundary at $X_0 = 0$.


In [3]:
def train_regression():
    print('=== Regression — Continuous CATE (Variance criterion) ===')
    np.random.seed(42)

    # ---- Data ----
    n, p = 700, 5
    X = np.random.randn(n, p)
    w = (np.random.rand(n) > 0.5).astype(int)
    tau_true = np.where(X[:, 0] > 0, 2.0, 0.5)
    y = tau_true * w + 0.5 * X[:, 1] + np.random.randn(n) * 0.3

    X_tr, X_te, y_tr, y_te, w_tr, w_te, tt_tr, tt_te = train_test_split(
        X, y, w, tau_true, test_size=0.3, random_state=42)
    feat = [f'X{i}' for i in range(p)]
    print(f'n_train={len(y_tr)}  treated={w_tr.sum()}  true ATE={tau_true.mean():.3f}')
    print(f'X_train[0]: {X_tr[0]}  y_train[0]: {y_tr[0]:.4f}')

    # ---- Model ----
    model = CausalTreeModel(
        task='regression',
        tree_config={
            'max_depth':         4,
            'min_samples_leaf':  15,
            'min_samples_treat': 5,
            'criterion':         'variance',
            'honest':            True,
        },
    )

    history = train_causal_tree(
        model, X_tr, y_tr, w_tr,
        X_val=X_te, y_val=y_te, w_val=w_te,
        metric='tau_risk', verbose=True,
    )
    print(model.summary())

    tau_te = model.predict(X_te)

    # ---- Metrics ----
    print('\nEvaluation metrics (test set):')
    for m, kw in [('mse',      {'tau_true': tt_te}),
                  ('mae',      {'tau_true': tt_te}),
                  ('rmse',     {'tau_true': tt_te}),
                  ('tau_risk', {'y': y_te, 'w': w_te}),
                  ('ate_bias', {'y': y_te, 'w': w_te})]:
        print(f'  {m:12s} = {compute_metric(m, tau_te, **kw):.4f}')

    # ---- VI / AIPW ----
    dr = doubly_robust_ate(tau_te, y_te, w_te)
    print(f'\nAIPW ATE = {dr["ate"]:.3f}  95% CI = [{dr["ci_lower"]:.3f}, {dr["ci_upper"]:.3f}]  p = {dr["pvalue"]:.4f}')
    print(f'  elbo_reg    = {compute_vi("elbo_reg",    tau_te, y=y_te, w=w_te):.4f}')
    print(f'  ltc_reg_loss= {compute_vi("ltc_reg_loss",tau_te, y=y_te, w=w_te):.4f}')

    # ---- Criterion comparison ----
    print('\nCriterion comparison (test tau_risk):')
    for crit in ['variance', 'mse', 'tau_risk']:
        m = CausalTreeModel(task='regression',
                            tree_config={'max_depth':4,'min_samples_leaf':15,'criterion':crit,'honest':True})
        m.fit(X_tr, y_tr, w_tr)
        print(f'  {crit:10s}  tau_risk={compute_metric("tau_risk", m.predict(X_te), y=y_te, w=w_te):.4f}')

    # ---- Depth sweep ----
    print('\nmax_depth sweep (test tau_risk):')
    depth_hist = []
    for d in range(1, 8):
        m = CausalTreeModel(task='regression',
                            tree_config={'max_depth':d,'min_samples_leaf':15,'honest':True})
        m.fit(X_tr, y_tr, w_tr)
        tr = compute_metric('tau_risk', m.predict(X_te), y=y_te, w=w_te)
        print(f'  depth={d}  tau_risk={tr:.4f}  leaves={m.get_n_leaves()}')
        depth_hist.append((d, tr))

    # ---- Plots (saved to disk — not interactive) ----
    print_tree(model, feature_names=feat)

    fig = plt.figure(figsize=(16, 9))
    plot_tree(model, feature_names=feat,
              title='Causal Tree — Regression (Variance Criterion)')
    plt.savefig('figures/regression_tree.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/regression_tree.png')

    fig = plt.figure(figsize=(9, 5))
    plot_cate_distribution(tau_te, task='regression',
                           title='CATE Distribution — Regression (test set)',
                           true_ate=tau_true.mean())
    plt.savefig('figures/regression_cate_dist.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/regression_cate_dist.png')

    fig = plt.figure(figsize=(10, 5))
    plot_treatment_effect_by_feature(X_te, tau_te, feature_idx=0,
                                     feature_name='X0', w=w_te, task='regression',
                                     title='CATE vs X0 — true boundary at X0=0')
    plt.savefig('figures/regression_cate_vs_feature.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/regression_cate_vs_feature.png')

    fig = plt.figure(figsize=(10, 7))
    plot_leaf_effects(model, X_te, y_te, w_te,
                      title='Per-Leaf CATE with 95% Bootstrap CI — Regression')
    plt.savefig('figures/regression_leaf_effects.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/regression_leaf_effects.png')

    fig = plt.figure(figsize=(14, 5))
    plot_decision_boundary(model, X_te, w_te, feat_x=0, feat_y=1,
                           feat_names=feat,
                           title='CATE Heatmap — Regression (X0 vs X1)')
    plt.savefig('figures/regression_heatmap.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/regression_heatmap.png')

    fig = plt.figure(figsize=(8, 4))
    plot_training_history(depth_hist, metric_name='tau_risk', xlabel='max_depth')
    plt.savefig('figures/regression_depth_sweep.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/regression_depth_sweep.png')

    # ---- Export + Arduino code ----
    export_to_json(model, 'json_model/regression_ct.json')
    generate_ino(
        'json_model/regression_ct.json',
        'arduino_code/regression_ino',
        board='esp32', task='regression',
    )
    return model


model_reg = train_regression()


=== Regression — Continuous CATE (Variance criterion) ===
n_train=490  treated=247  true ATE=1.282
X_train[0]: [-1.4575515  -0.30920908 -0.75215641  0.31917451  1.34045045]  y_train[0]: -0.2701
  [train] TAU_RISK = 3.530141
  [val  ] TAU_RISK = 3.747943
  depth=1  leaves=2
CausalTreeModel
  task        : regression
  criterion   : variance
  honest      : True
  max_depth   : 4
  depth (fit) : 1
  n_leaves    : 2

  Leaf estimates (2 leaves):
    Leaf  1: τ̂=+0.2675  T=55  C=43
    Leaf  2: τ̂=+1.8239  T=80  C=67

Evaluation metrics (test set):
  mse          = 0.0984
  mae          = 0.2388
  rmse         = 0.3137
  tau_risk     = 3.7479
  ate_bias     = 0.1022

AIPW ATE = 1.252  95% CI = [1.025, 1.478]  p = 0.0000
  elbo_reg    = 3.7499
  ltc_reg_loss= 0.6244

Criterion comparison (test tau_risk):
  variance    tau_risk=3.7479
  mse         tau_risk=3.9737
  tau_risk    tau_risk=3.9168

max_depth sweep (test tau_risk):
  depth=1  tau_risk=3.7479  leaves=2
  depth=2  tau_risk=3.7479  

c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:281: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/regression_tree.png


c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:377: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/regression_cate_dist.png


...

Saved figures/regression_leaf_effects.png


c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:519: UserWarning: The following kwargs were not used by contour: 'lw'
  axes[1].contour(xx, yy, policy, levels=[0.5], colors='black', lw=1.5)
c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:528: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/regression_heatmap.png


c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:692: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/regression_depth_sweep.png
Model exported → json_model/regression_ct.json
Generated in  : arduino_code/regression_ino  (board: esp32, task: regression)
n_features    : 1
Expected output (Python): 1.823891138218874


---
## 3. Example 2 — Binary Classification: Risk Difference

The outcome $Y \in \{0,1\}$ represents a binary event (e.g. recovery).  
The treatment effect is a risk difference: $\hat\tau = P(Y=1|T) - P(Y=1|C)$.  
We use the **Tau-Risk criterion**, which is more robust when propensity varies.


In [4]:
def train_binary():
    print('=== Binary Classification — Risk Difference (Tau-Risk criterion) ===')
    np.random.seed(7)

    # ---- Data ----
    n, p = 800, 5
    X = np.random.randn(n, p)
    e = 1 / (1 + np.exp(-0.5 * X[:, 0]))       # propensity varies with X0
    w = (np.random.rand(n) < e).astype(int)
    tau_true = np.where(X[:, 0] > 0.5, 0.30, 0.05)
    p_base   = 1 / (1 + np.exp(-0.4 * X[:, 1]))
    p_obs    = w * np.clip(p_base + tau_true, 0, 1) + (1 - w) * p_base
    y        = (np.random.rand(n) < p_obs).astype(int)

    X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
        X, y, w, test_size=0.3, random_state=7)
    feat = [f'X{i}' for i in range(p)]
    print(f'n_train={len(y_tr)}  treated={w_tr.sum()}  true ATE={tau_true.mean():.3f}')
    print(f'X_train[0]: {X_tr[0]}  y_train[0]: {y_tr[0]}')

    # ---- Model ----
    model = CausalTreeModel(
        task='binary',
        tree_config={
            'max_depth':         3,
            'min_samples_leaf':  20,
            'min_samples_treat': 5,
            'criterion':         'tau_risk',
            'honest':            True,
        },
    )

    train_causal_tree(model, X_tr, y_tr, w_tr,
                      X_val=X_te, y_val=y_te, w_val=w_te,
                      metric='tau_risk', verbose=True)
    print(model.summary())

    tau_bin = model.predict(X_te)

    # ---- Metrics ----
    print('\nEvaluation metrics (test set):')
    for m, kw in [('tau_risk',        {'y': y_te.astype(float), 'w': w_te}),
                  ('ate_bias',        {'y': y_te.astype(float), 'w': w_te}),
                  ('bce',             {'y': y_te.astype(float), 'w': w_te}),
                  ('accuracy',        {'y': y_te, 'w': w_te}),
                  ('risk_diff_error', {'y': y_te.astype(float), 'w': w_te})]:
        print(f'  {m:20s} = {compute_metric(m, tau_bin, **kw):.4f}')

    # ---- VI ----
    dr = doubly_robust_ate(tau_bin, y_te.astype(float), w_te)
    print(f'\nAIPW ATE = {dr["ate"]:.3f}  95% CI = [{dr["ci_lower"]:.3f}, {dr["ci_upper"]:.3f}]  p = {dr["pvalue"]:.4f}')
    print(f'  elbo_binary    = {compute_vi("elbo_binary",    tau_bin, y=y_te.astype(float), w=w_te):.4f}')
    print(f'  policy_loss_bin= {compute_vi("policy_loss_bin",tau_bin, y=y_te.astype(float), w=w_te):.4f}')

    # ---- Criterion comparison ----
    print('\nCriterion comparison (test tau_risk):')
    for crit in ['variance', 'mse', 'tau_risk']:
        m = CausalTreeModel(task='binary',
                            tree_config={'max_depth':3,'min_samples_leaf':20,'criterion':crit,'honest':True})
        m.fit(X_tr, y_tr, w_tr)
        tr  = compute_metric('tau_risk',       m.predict(X_te), y=y_te.astype(float), w=w_te)
        rde = compute_metric('risk_diff_error',m.predict(X_te), y=y_te.astype(float), w=w_te)
        print(f'  {crit:10s}  tau_risk={tr:.4f}  risk_diff_err={rde:.4f}')

    # ---- Plots (saved to disk) ----
    print_tree(model, feature_names=feat)

    fig = plt.figure(figsize=(16, 9))
    plot_tree(model, feature_names=feat,
              title='Causal Tree — Binary Classification (Tau-Risk Criterion)')
    plt.savefig('figures/binary_tree.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/binary_tree.png')

    fig = plt.figure(figsize=(9, 5))
    plot_cate_distribution(tau_bin, task='binary',
                           title='Risk Difference Distribution — Binary (test set)',
                           true_ate=tau_true.mean())
    plt.savefig('figures/binary_cate_dist.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/binary_cate_dist.png')

    fig = plt.figure(figsize=(10, 5))
    plot_treatment_effect_by_feature(X_te, tau_bin, feature_idx=0,
                                     feature_name='X0', w=w_te, task='binary',
                                     title='Risk Difference vs X0 — true boundary at X0=0.5')
    plt.savefig('figures/binary_rd_vs_feature.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/binary_rd_vs_feature.png')

    fig = plt.figure(figsize=(10, 6))
    plot_leaf_effects(model, X_te, y_te.astype(float), w_te,
                      title='Per-Leaf Risk Difference with 95% Bootstrap CI')
    plt.savefig('figures/binary_leaf_effects.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/binary_leaf_effects.png')

    fig = plt.figure(figsize=(14, 5))
    plot_decision_boundary(model, X_te, w_te, feat_x=0, feat_y=1,
                           feat_names=feat,
                           title='Risk Difference Heatmap — Binary (X0 vs X1)')
    plt.savefig('figures/binary_heatmap.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/binary_heatmap.png')

    # ---- Export + Arduino code ----
    export_to_json(model, 'json_model/binary_ct.json')
    generate_ino(
        'json_model/binary_ct.json',
        'arduino_code/binary_ino',
        board='esp32', task='binary',
    )
    return model


model_bin = train_binary()


=== Binary Classification — Risk Difference (Tau-Risk criterion) ===
n_train=560  treated=276  true ATE=0.129
X_train[0]: [ 0.49134459  0.82264717 -1.48803104 -0.51291001  0.79079868]  y_train[0]: 1
  [train] TAU_RISK = 2.208271
  [val  ] TAU_RISK = 2.305200
  depth=1  leaves=2
CausalTreeModel
  task        : binary
  criterion   : tau_risk
  honest      : True
  max_depth   : 3
  depth (fit) : 1
  n_leaves    : 2

  Leaf estimates (2 leaves):
    Leaf  1: τ̂=+0.1991  T=31  C=41
    Leaf  2: τ̂=+0.1352  T=108  C=100

Evaluation metrics (test set):
  tau_risk             = 2.3052
  ate_bias             = 0.0633
  bce                  = 0.6800
  accuracy             = 0.5375
  risk_diff_error      = 0.0633

AIPW ATE = 0.089  95% CI = [-0.037, 0.214]  p = 0.1656
  elbo_binary    = 0.7233
  policy_loss_bin= -0.6202

Criterion comparison (test tau_risk):
  variance    tau_risk=2.3052  risk_diff_err=0.0633
  mse         tau_risk=2.2871  risk_diff_err=0.0542
  tau_risk    tau_risk=2.3052  ris

c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:281: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/binary_tree.png


c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:377: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/binary_cate_dist.png


...

Saved figures/binary_rd_vs_feature.png


c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:612: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/binary_leaf_effects.png


c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:519: UserWarning: The following kwargs were not used by contour: 'lw'
  axes[1].contour(xx, yy, policy, levels=[0.5], colors='black', lw=1.5)
c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:528: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/binary_heatmap.png
Model exported → json_model/binary_ct.json
Generated in  : arduino_code/binary_ino  (board: esp32, task: binary)
n_features    : 2
Expected output (Python): 0.13518518518518519


---
## 4. Example 3 — Multiclass: Per-Class Treatment Effect Shifts

The outcome $Y \in \{0, 1, 2\}$ represents disease severity (Mild / Moderate / Severe).  
The CATE becomes a $K$-vector: $\hat\tau_k = P(Y=k|T) - P(Y=k|C)$.  
The predicted treatment-induced class is $\arg\max_k \hat\tau_k$.


In [5]:
def train_multiclass():
    print('=== Multiclass — Disease Severity K=3 (Variance criterion) ===')
    np.random.seed(0)

    # ---- Data ----
    n, p, K = 900, 6, 3
    X = np.random.randn(n, p)
    w = (np.random.rand(n) > 0.5).astype(int)

    logits_base = np.column_stack([-0.5*X[:,1], 0.3*X[:,1], 0.4*X[:,2]])
    exp_b = np.exp(logits_base - logits_base.max(1, keepdims=True))
    p_base = exp_b / exp_b.sum(1, keepdims=True)

    shift = np.zeros((n, K))
    mask_hi = X[:, 0] > 0
    shift[mask_hi,  2] =  0.35;  shift[mask_hi,  0] = -0.20
    shift[~mask_hi, 1] =  0.20;  shift[~mask_hi, 0] = -0.10

    p_treat = np.clip(p_base + shift, 0, 1)
    p_treat /= p_treat.sum(1, keepdims=True)
    p_obs    = w[:,None] * p_treat + (1 - w[:,None]) * p_base

    cumul = p_obs.cumsum(1)
    y     = (np.random.rand(n)[:,None] > cumul).sum(1).astype(int)

    X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
        X, y, w, test_size=0.3, random_state=0)
    feat   = [f'X{i}' for i in range(p)]
    cnames = ['Mild (0)', 'Moderate (1)', 'Severe (2)']
    print(f'n_train={len(y_tr)}  treated={w_tr.sum()}')
    print(f'class dist: {np.bincount(y_tr)}')
    print(f'X_train[0]: {X_tr[0]}  y_train[0]: {y_tr[0]}')

    # ---- Model ----
    model = CausalTreeModel(
        task='multiclass',
        n_classes=K,
        tree_config={
            'max_depth':         4,
            'min_samples_leaf':  20,
            'min_samples_treat': 5,
            'criterion':         'variance',
            'honest':            True,
        },
    )

    train_causal_tree(model, X_tr, y_tr, w_tr,
                      X_val=X_te, y_val=y_te, w_val=w_te,
                      metric='accuracy_multi', verbose=True)
    print(model.summary())

    labels = model.predict(X_te)
    proba  = model.predict_proba(X_te)

    # ---- Metrics ----
    print('\nEvaluation metrics (test set):')
    for m, arg, kw in [
            ('accuracy_multi',  labels, {'y': y_te, 'w': w_te}),
            ('macro_ate_error', proba,  {'y': y_te, 'w': w_te}),
            ('scce',            labels, {'y': y_te, 'w': w_te})]:
        print(f'  {m:20s} = {compute_metric(m, arg, **kw):.4f}')

    # ---- VI ----
    print(f'\n  elbo_multi        = {compute_vi("elbo_multi",        proba, y=y_te, w=w_te):.4f}')
    print(f'  policy_loss_multi = {compute_vi("policy_loss_multi", proba, y=y_te, w=w_te):.4f}')

    # ---- Per-class AIPW ATEs ----
    print('\nPer-class AIPW ATE:')
    for k in range(K):
        y_k = (y_te == k).astype(float)
        dr  = doubly_robust_ate(proba[:, k], y_k, w_te)
        print(f'  {cnames[k]:15s}  ATE={dr["ate"]:+.3f}  '
              f'CI=[{dr["ci_lower"]:+.3f}, {dr["ci_upper"]:+.3f}]  p={dr["pvalue"]:.4f}')

    # ---- Criterion comparison ----
    print('\nCriterion comparison (accuracy_multi on test):')
    for crit in ['variance', 'mse', 'tau_risk']:
        m = CausalTreeModel(task='multiclass', n_classes=K,
                            tree_config={'max_depth':4,'min_samples_leaf':20,
                                         'criterion':crit,'honest':True})
        m.fit(X_tr, y_tr, w_tr)
        acc = compute_metric('accuracy_multi', m.predict(X_te), y=y_te, w=w_te)
        mae = compute_metric('macro_ate_error', m.predict_proba(X_te), y=y_te, w=w_te)
        print(f'  {crit:10s}  accuracy={acc:.4f}  macro_ate_err={mae:.4f}')

    # ---- Depth sweep ----
    print('\nmax_depth sweep (accuracy_multi on test):')
    depth_hist = []
    for d in range(1, 7):
        m = CausalTreeModel(task='multiclass', n_classes=K,
                            tree_config={'max_depth':d,'min_samples_leaf':20,'honest':True})
        m.fit(X_tr, y_tr, w_tr)
        acc = compute_metric('accuracy_multi', m.predict(X_te), y=y_te, w=w_te)
        print(f'  depth={d}  accuracy={acc:.4f}  leaves={m.get_n_leaves()}')
        depth_hist.append((d, -acc))

    # ---- Plots (saved to disk) ----
    print_tree(model, feature_names=feat, class_names=cnames)

    fig = plt.figure(figsize=(16, 9))
    plot_tree(model, feature_names=feat, class_names=cnames,
              title='Causal Tree — Multiclass K=3 (Variance Criterion)')
    plt.savefig('figures/multiclass_tree.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/multiclass_tree.png')

    fig = plt.figure(figsize=(13, 5))
    plot_multiclass_effects(model, class_names=cnames,
                            title='Per-Leaf Per-Class Treatment Effect τ̂_k')
    plt.savefig('figures/multiclass_leaf_heatmap.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/multiclass_leaf_heatmap.png')

    fig = plt.figure(figsize=(10, 5))
    plot_treatment_effect_by_feature(
        X_te, labels.astype(float), feature_idx=0,
        feature_name='X0', w=w_te, task='multiclass',
        title='Predicted Treatment Class vs X0')
    plt.savefig('figures/multiclass_class_vs_feature.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/multiclass_class_vs_feature.png')

    fig = plt.figure(figsize=(14, 5))
    plot_decision_boundary(model, X_te, w_te, feat_x=0, feat_y=1,
                           feat_names=feat, class_names=cnames,
                           title='Predicted Class Heatmap (X0 vs X1)')
    plt.savefig('figures/multiclass_heatmap.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/multiclass_heatmap.png')

    fig = plt.figure(figsize=(8, 4))
    plot_training_history([(d, -v) for d, v in depth_hist],
                          metric_name='accuracy_multi', xlabel='max_depth')
    plt.savefig('figures/multiclass_depth_sweep.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved figures/multiclass_depth_sweep.png')

    # ---- Export + Arduino code ----
    export_to_json(model, 'json_model/multiclass_ct.json')
    generate_ino(
        'json_model/multiclass_ct.json',
        'arduino_code/multiclass_ino',
        board='esp32', task='multiclass',
    )
    return model


model_mc = train_multiclass()


=== Multiclass — Disease Severity K=3 (Variance criterion) ===
n_train=630  treated=321
class dist: [150 244 236]
X_train[0]: [-0.46886419 -2.20144129  0.1993002  -0.05060354 -0.51751904 -0.97882986]  y_train[0]: 2
  [train] ACCURACY_MULTI = 0.457944
  [val  ] ACCURACY_MULTI = 0.338583
  depth=3  leaves=6
CausalTreeModel
  task        : multiclass
  criterion   : variance
  honest      : True
  max_depth   : 4
  depth (fit) : 3
  n_leaves    : 6
  n_classes   : 3

  Leaf estimates (6 leaves):
    Leaf  1: τ̂=[-0.208,-0.315, 0.523]  T=10  C=13
    Leaf  2: τ̂=[-0.148,-0.009, 0.157]  T=36  C=35
    Leaf  3: τ̂=[0.,0.,0.]  T=8  C=12
    Leaf  4: τ̂=[-0.122,-0.014, 0.136]  T=66  C=47
    Leaf  5: τ̂=[-0.286, 0.   , 0.286]  T=7  C=7
    Leaf  6: τ̂=[-0.178,-0.056, 0.234]  T=34  C=40

Evaluation metrics (test set):
  accuracy_multi       = 0.3386
  macro_ate_error      = 0.1089
  scce                 = 1.2129

  elbo_multi        = 1.1032
  policy_loss_multi = -0.3386

Per-class AIPW ATE:
  

c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:281: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/multiclass_tree.png


c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:662: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/multiclass_leaf_heatmap.png


...

Saved figures/multiclass_class_vs_feature.png


c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:528: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/multiclass_heatmap.png


c:\Users\thomm\OneDrive\Desktop\Repositorios\TinyML\36_CT\utils.py:692: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Saved figures/multiclass_depth_sweep.png
Model exported → json_model/multiclass_ct.json
Generated in  : arduino_code/multiclass_ino  (board: esp32, task: multiclass)
n_features    : 5
Expected output (Python): 2


---
## 5. Quick Reference — Architecture & Hyperparameters

### CausalTreeModel configuration

```python
CausalTreeModel(
    task         = 'regression',   # 'regression' | 'binary' | 'multiclass'
    n_classes    = None,           # required for multiclass
    random_state = 42,
    tree_config  = {
        'max_depth':         5,        # tree depth
        'min_samples_leaf':  20,       # min samples per leaf (structure half)
        'min_samples_treat': 5,        # min treated units per child
        'criterion':         'variance', # 'variance' | 'mse' | 'tau_risk'
        'honest':            True,     # separate structure / estimation halves
        'n_features':        None,     # None | 'sqrt' | 'log2' | int
    },
)
```

### Loss / metric functions (`losses.py`)

| Task | Key | Requires |
|------|-----|----------|
| All | `tau_risk` | `y`, `w` |
| All | `ate_bias` | `y`, `w` |
| Regression | `mse`, `mae`, `rmse` | `tau_true` |
| Binary | `bce`, `accuracy`, `risk_diff_error` | `y`, `w` |
| Multiclass | `scce`, `accuracy_multi` | `y`, `w` |
| Multiclass | `macro_ate_error` | `y`, `w` (pass `predict_proba` output) |

### VI / Policy methods (`vi.py`)

| Key | Task | Description |
|-----|------|-------------|
| `dr_ate` | All | Doubly-robust AIPW ATE |
| `elbo_reg` | Regression | AIPW-MSE + τ² regularisation |
| `ltc_reg_loss` | Regression | MAE + mean\|τ̂\| regularisation |
| `elbo_binary` | Binary | AIPW-BCE + τ² regularisation |
| `policy_loss_bin` | Binary | Negative IPW policy value |
| `elbo_multi` | Multiclass | Per-class AIPW-CCE + Frobenius regularisation |
| `policy_loss_multi` | Multiclass | Negative IPW policy value (K classes) |

### Arduino export

```python
from cpp_generator import generate_ino

generate_ino(
    'json_model/regression_ct.json',
    'arduino_code/regression_ino',
    board='esp32',    # 'avr' | 'esp32'
    task='regression',
)
# Produces: CTModel.h  +  sketch.ino
```
